# 🎬 ĐẾM trên VIDEO THẬT — DỄ → KHÓ (Colab & Kaggle)

Chạy **đếm** cho MỌI bài toán, mỗi video test từ **query DỄ → KHÓ**:
- 🚗 **Xe**: `car`/`vehicle`/`truck`/`bus` → `a white car`/`a red car`/`a large truck`
- 🚶 **Người**: `person` → `a person wearing a backpack`/`a person in white`
- 📦 **Dây chuyền**: `object`/`box`/`tomato` → `a cardboard box`/`a red tomato`/`a cluster of tomatoes`

**Đếm CHÍNH XÁC**: vạch đã đặt lại đúng hướng dòng chảy (phân tích optical-flow), chạy
`stride=1` (KHÔNG bỏ frame → track không đứt), đếm theo **vật CẮT VẠCH** + **tracks** (số vật khác nhau).
Mỗi lần đếm **lưu video annotate** để xem lại.

> Cần **GPU** (Colab: Runtime→T4 · Kaggle: Settings→Accelerator→GPU). Dây chuyền dùng
> LocateAnything-3B (chậm ~vài giây/frame) → để `--max-frames` vừa phải.

## 1) Cài đặt + tải code (tự nhận Colab/Kaggle)

In [ ]:
import os
if os.path.isdir('/kaggle/working'):      WORK = '/kaggle/working'
elif os.path.isdir('/content'):           WORK = '/content'
else:                                      WORK = os.path.abspath('.')
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
print('📂 WORK =', WORK)

REPO   = 'https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git'
BRANCH = 'claude/locate-anything-test-suite-xwju2f'          # nhánh CÓ code mới nhất (vạch đã sửa, service, detector đúng)
if not os.path.isdir(f'{WORK}/VisionOS/.git'):
    os.system(f'git clone -q {REPO} {WORK}/VisionOS')
os.chdir(f'{WORK}/VisionOS')
os.system(f'git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset --hard -q origin/{BRANCH}')
os.chdir(f'{WORK}/VisionOS/VisionOS')
print('📁 cwd  =', os.getcwd(), '| nhánh =', 'claude/locate-anything-test-suite-xwju2f')
os.system("pip install -q ultralytics 'supervision>=0.21' opencv-python-headless")

import torch
print('🖥️  GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else '❌ CHƯA BẬT GPU! Colab: Runtime→Change runtime type→T4 · Kaggle: Settings→Accelerator→GPU')

## 2) ✅ Kiểm tra video tải được (nhanh, không cần model)

In [ ]:
!python run_scenarios.py --download-only

## 3) ⭐ XEM TRƯỚC vạch/vùng (đã sửa) trên frame thật
Vàng = **vạch** cắt, xanh = **vùng**, lưới = %. Nhìn xem vạch có nằm ĐÚNG chỗ vật đi qua
không (kiện hàng: vạch **dọc**; cà chua: vạch **ngang**). Muốn đổi → báo toạ độ % hoặc dùng
`--line 'x1,y1,x2,y2'`.

In [ ]:
import os, glob
from IPython.display import Image, display, Markdown
WORK = globals().get('WORK') or ('/kaggle/working' if os.path.isdir('/kaggle/working') else '/content')
!python run_scenarios.py --preview {WORK}/prev
allp = f'{WORK}/prev/_ALL.jpg'
if os.path.exists(allp):
    display(Markdown('### 🧩 TỔNG HỢP tất cả trường hợp')); display(Image(filename=allp, width=940))

## 4) ⏳ Tải model LocateAnything-3B (cho dây chuyền, chạy 1 lần ~6GB)
Xe/người dùng YOLO (không cần bước này); dây chuyền (thùng/cà chua open-vocab) cần model 3B.

In [ ]:
from huggingface_hub import snapshot_download
print("✅ Model ở cache:", snapshot_download("nvidia/LocateAnything-3B"))

## 5) ⭐ ĐẾM TẤT CẢ — query DỄ (chính xác, lưu video)
Mỗi bài chạy 1 prompt cơ bản, `stride=1` (đếm chuẩn), lưu video annotate vào `scen_out/`.
- Xe/Người: **YOLO** (nhanh → nhiều frame).
- Dây chuyền: **LocateAnything** (chậm → `--max-frames 80`; tăng nếu muốn kỹ hơn).

In [ ]:
WORK = globals().get('WORK') or '/content'
# 🚗 XE + 🚶 NGƯỜI — YOLO nhanh, đếm 300 frame cho chuẩn
!python run_scenarios.py --task vehicles --max-frames 300 --save-dir {WORK}/scen_out
!python run_scenarios.py --task people   --max-frames 300 --save-dir {WORK}/scen_out

In [ ]:
WORK = globals().get('WORK') or '/content'
# 📦 DÂY CHUYỀN — LocateAnything, vạch ĐÃ SỬA (con lăn=dọc x=50, belt=ngang y=50, cà chua=ngang y=70)
# stride=1 = KHÔNG bỏ frame → track không đứt → đếm sát hơn. LA chậm → ~10-15 phút.
!python run_scenarios.py --task conveyor --max-frames 80 --save-dir {WORK}/scen_out

## 6) ⭐ ĐẾM query DỄ → KHÓ (bài test chính)
Đếm LẦN LƯỢT mỗi query trong cột `queries` (DỄ→KHÓ) trên từng video. Câu chuyện đếm:
**`car` (mọi xe) → `a red car` (chỉ xe đỏ)** ⇒ số đếm GIẢM dần khi query hẹp hơn — đó là
sức mạnh open-vocab (đếm CÓ ĐIỀU KIỆN) mà YOLO không làm được.

⚠️ CHẬM: mỗi query = 1 lượt đếm cả video. Để `--max-frames` NHỎ. Bỏ `--only` để chạy hết.

In [ ]:
WORK = globals().get('WORK') or '/content'
# 📦 Dây chuyền: đếm con lăn + cà chua qua các prompt dễ→khó (bỏ --only để chạy cả 5 video)
!python run_scenarios.py --task conveyor --only rollers --all-queries --max-frames 40 --save-dir {WORK}/scen_q
!python run_scenarios.py --task conveyor --only tomato  --all-queries --max-frames 40 --save-dir {WORK}/scen_q

In [ ]:
WORK = globals().get('WORK') or '/content'
# 🚗 Xe: đếm mọi xe → theo MÀU/LOẠI (car→a white car→a red car→a large truck)
!python run_scenarios.py --task vehicles --only "giao lộ" --all-queries --max-frames 150 --save-dir {WORK}/scen_q

## 7) 🎥 Xem / tải video output
Video annotate (vạch/vùng + box + nhãn + số đếm) trong `scen_out/` (query dễ) và `scen_q/`
(dễ→khó). Kaggle: tải ở panel **Output**. Dưới đây phát thử vài video (đổi sang H.264).

In [ ]:
import glob, os
from IPython.display import Video, display, Markdown
WORK = globals().get('WORK') or ('/kaggle/working' if os.path.isdir('/kaggle/working') else '/content')
vids = sorted(glob.glob(f'{WORK}/scen_out/**/*.mp4', recursive=True)) + \
       sorted(glob.glob(f'{WORK}/scen_q/**/*.mp4', recursive=True))
print(f'{len(vids)} video output:')
for p in vids: print('  ', p)
for src in vids[:3]:                      # phát 3 video đầu
    dst = src.replace('.mp4', '_h264.mp4')
    os.system(f'ffmpeg -y -loglevel error -i "{src}" -vcodec libx264 -pix_fmt yuv420p "{dst}"')
    display(Markdown(f'**{os.path.basename(src)}**')); display(Video(dst, embed=True, width=680))

### 📖 Đọc số đếm (quan trọng — đừng nhầm)
- **IN/OUT/total** = số vật ĐI QUA VẠCH (mỗi vật đếm 1 lần khi CẮT vạch) → "đếm vào/ra".
- **tracks** = TỔNG số vật KHÁC NHAU thấy trong video (≈ "có bao nhiêu xe/thùng/quả"; hơi
  DƯ do track đứt-nối). Đây mới là con số "nhiều vật" nếu bạn muốn tổng.
- **trong_vùng / đỉnh_vùng** = số vật ĐANG trong vùng (hiện tại / đông nhất) — bài ĐẾM VÙNG.
- **det/frame** = TB vật/khung — đo sức DETECT của model (cao = nhận diện tốt).

**Đếm ra 0?** Không phải model mù — hầu hết do **VẠCH đặt lệch dòng đi**. Xem lại ở mục (3),
đổi vạch bằng `--line 'x1,y1,x2,y2'` (%). Cảnh **đóng gói tay** (belt) vốn ít vật cắt vạch —
dùng cột `tracks` hoặc bài **ĐẾM VÙNG** (cà chua) cho cảnh dày/nhanh.